In [2]:
from huggingface_hub import list_datasets

datasets = list(list_datasets())

print(f"There are {len(datasets)} datasets available")
print(datasets[:10])

There are 1002731 datasets available
[DatasetInfo(id='ADSKAILab/Zero-To-CAD-1m', author='ADSKAILab', sha='09dbd1805a5e73a2757f380b93042b8089cd4f3f', created_at=datetime.datetime(2026, 4, 11, 16, 7, 48, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2026, 5, 3, 14, 11, 21, tzinfo=datetime.timezone.utc), private=False, gated=False, disabled=False, downloads=20111, downloads_all_time=None, likes=103, paperswithcode_id=None, tags=['task_categories:text-to-3d', 'task_categories:image-to-3d', 'language:en', 'language:code', 'license:apache-2.0', 'size_categories:100K<n<1M', 'format:parquet', 'modality:tabular', 'modality:text', 'library:datasets', 'library:dask', 'library:polars', 'library:mlcroissant', 'arxiv:2604.24479', 'region:us', 'CAD', 'CadQuery', 'synthetic-data', 'construction-sequence', 'parametric-CAD', '3D-generation', 'agentic-AI', 'code-generation'], trending_score=68, card_data=None, siblings=None, xet_enabled=None), DatasetInfo(id='TuringEnterprises/Open-MM-RL

In [3]:
from datasets import load_dataset
emotions = load_dataset("emotion")

c:\Users\PP\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PP\.cache\huggingface\hub\datasets--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

In [ ]:
dataset_url = "https://www.dropbox.com/s/1pzkadrvffbqw6o/train.txt"
!wget {dataset_url}


Téléchargement terminé


In [7]:
emotions_local = load_dataset("csv", data_files="train.txt", sep=";",
 names=["text", "label"])

Generating train split: 0 examples [00:00, ? examples/s]Failed to read file 'C:\Users\PP\Music\tutoNlptransformer\textclassification\train.txt' with error <class 'pandas.errors.ParserError'>: Error tokenizing data. C error: Expected 2 fields in line 6, saw 3

Generating train split: 0 examples [00:00, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [ ]:
dataset_url = "https://www.dropbox.com/s/1pzkadrvffbqw6o/train.txt?dl=1"
emotions_remote = load_dataset("csv", data_files=dataset_url, sep=";",
 names=["text", "label"])

In [ ]:
import pandas as pd
emotions.set_format(type="pandas")
df = emotions["train"][:]
df.head()


In [ ]:
def label_int2str(row):
 return emotions["train"].features["label"].int2str(row)

df["label_name"] = df["label"].apply(label_int2str)
df.head()


In [ ]:
import matplotlib.pyplot as plt

df["label_name"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of Classes")
plt.show()


In [ ]:
df["Words Per Tweet"] = df["text"].str.split().apply(len)
df.boxplot("Words Per Tweet", by="label_name", grid=False,
 showfliers=False, color="black")
plt.suptitle("")
plt.xlabel("")
plt.show()

In [ ]:
tokenized_text = text.split()
print(tokenized_text)


In [8]:
from transformers import AutoTokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

c:\Users\PP\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PP\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [9]:
from transformers import DistilBertTokenizer
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(model_ckpt)

In [10]:
encoded_text = tokenizer(text)
print(encoded_text)


NameError: name 'text' is not defined

In [ ]:
def tokenize(batch):
 return tokenizer(batch["text"], padding=True, truncation=True)

In [ ]:
emotions_encoded = emotions.map(tokenize, batched=True, batch_size=None)

In [11]:
import torch
from transformers import AutoModel
model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_ckpt).to(device)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [12]:
text = "this is a test"
inputs = tokenizer(text, return_tensors="pt")
print(f"Input tensor shape: {inputs['input_ids'].size()}")

Input tensor shape: torch.Size([1, 6])


In [ ]:
inputs = {k:v.to(device) for k,v in inputs.items()}
with torch.no_grad():
 outputs = model(**inputs)
print(outputs)